# load_openalex_work_authorships

Prototipo del nodo `load_openalex_work_authorships` del pipeline `load_openalex`. No guarda datasets.


In [ ]:
import pandas as pd
from pandas import json_normalize

%load_ext kedro.ipython


In [ ]:
df_work_raw = catalog.load('raw/openalex/work/parquet/work_dev')
df_work_raw.head(2)


In [ ]:
def _select_with_metadata(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = _add_openalex_extracted_metadata(df)
    return df.loc[:, [*columns, *_EXTRACTED_META_COLS]].copy()


In [ ]:
def _stringify_object_columns(
    df: pd.DataFrame,
    exclude_columns: list[str] | None = None,
) -> pd.DataFrame:
    exclude_columns = set(exclude_columns or [])
    for column in df.columns:
        if column in exclude_columns:
            continue
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].where(df[column].notna(), pd.NA).astype("string")
    return df


In [ ]:
def _serialize_nested_value(value):
    if value is None or value is pd.NA:
        return value
    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


In [ ]:
def _serialize_nested_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(_serialize_nested_value)
    return df


In [ ]:
def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df["extract_datetime"] = pd.to_datetime(df["extract_datetime"], errors="coerce")
    df["_extract_datetime"] = pd.to_datetime(df["_extract_datetime"], errors="coerce")
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date
    return df


In [ ]:
def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = _serialize_nested_columns(df)
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    load_datetime = pd.to_datetime(load_datetime)
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openalex_work_authorships(df_work_raw):

    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    # Seleccionar las columnas necesarias y convertir los tipos de datos
    df_work2authorships = df_work_raw[['id', 'authorships', '_filter_param', '_filter_value', '_extract_datetime']].convert_dtypes()
    df_work2authorships.rename(columns={"id": "work_id"}, inplace=True)

    # Expandir la lista de authorships
    df_work2authorships_exploded = df_work2authorships.explode('authorships', ignore_index=True)

    # Normalizar la información de authorships
    df_authorships_norm = pd.json_normalize(df_work2authorships_exploded['authorships'])
    df_authorships_norm.rename(
        columns={
            "author.id": "author_id",
            "author.display_name": "author_display_name",
            "author.orcid": "author_orcid",
        },
        inplace=True,
    )

    # Combinar work_id con la información normalizada de authorships
    df_work2authorships = df_work2authorships_exploded[
        ['work_id', '_filter_param', '_filter_value', '_extract_datetime']
    ].join(df_authorships_norm)

    # Asegurar presencia de columnas aunque no vengan en todos los payloads
    for col in ["author_id", "author_display_name", "author_orcid", "author_position", "institutions"]:
        if col not in df_work2authorships.columns:
            df_work2authorships[col] = pd.NA

    # Extraer la relación work-author
    df_work2author = df_work2authorships[
        [
            'work_id',
            'author_id',
            'author_display_name',
            'author_orcid',
            'author_position',
            '_filter_param',
            '_filter_value',
            '_extract_datetime',
        ]
    ]

    # Expandir la lista de instituciones asociadas a cada autor
    df_work2institution_exploded = df_work2authorships.explode('institutions', ignore_index=True)

    # Normalizar la información de instituciones
    df_institution_norm = pd.json_normalize(df_work2institution_exploded['institutions'])
    df_institution_norm.drop(columns=['lineage'], errors='ignore', inplace=True)

    # Combinar author_id con la información normalizada de instituciones
    df_author2institution = df_work2institution_exploded[
        ['author_id', '_filter_param', '_filter_value', '_extract_datetime']
    ].join(df_institution_norm)

    # Combinar work_id con la información normalizada de instituciones
    df_work2institution = df_work2institution_exploded[
        ['work_id', '_filter_param', '_filter_value', '_extract_datetime']
    ].join(df_institution_norm)

    df_work2author = _add_openalex_loaded_metadata(df_work2author)
    df_work2institution = _add_openalex_loaded_metadata(df_work2institution)
    df_author2institution = _add_openalex_loaded_metadata(df_author2institution)

    return df_work2author, df_work2institution, df_author2institution


In [ ]:
df_work2author, df_work2institution, df_author2institution = load_openalex_work_authorships(df_work_raw)


In [ ]:
pd.DataFrame([
    {'dataset': 'df_work2author', 'rows': len(df_work2author), 'columns': len(df_work2author.columns)},
    {'dataset': 'df_work2institution', 'rows': len(df_work2institution), 'columns': len(df_work2institution.columns)},
    {'dataset': 'df_author2institution', 'rows': len(df_author2institution), 'columns': len(df_author2institution.columns)},
])


In [ ]:
df_work2author.head(2)
